# 🔬 SÍNTESE 1 — FASE EXPLORATÓRIA
## Notebooks 1–10 | Evidências e Métricas Reais

> Todos os valores abaixo foram medidos experimentalmente. pAUC@0.1 calculado
> a partir das curvas ROC reais de cada notebook.

---

### Protocolo de Avaliação
- **Métrica principal**: pAUC@FPR0.1 — área parcial sob a ROC na região de baixo falso positivo
- **Limite mínimo**: 0.80 (critério de aprovação)
- **Latência máxima**: 50 ms | **Memória máxima**: 4 MB
- **Validação**: LOSO (Leave-One-Source-Out)


In [ ]:
import json, sys, os
sys.path.insert(0, '..')
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

METRICS_DIR = Path('experiments_results/metrics')
FIGURES_DIR = Path('experiments_results/figures')

def load(nb_id):
    p = METRICS_DIR / f'{nb_id}_results.json'
    return json.load(open(p)) if p.exists() else {}

# Carrega resultados
nb01 = load('nb01'); nb02 = load('nb02'); nb03 = load('nb03')
nb04 = load('nb04'); nb05 = load('nb05'); nb06 = load('nb06')
nb07 = load('nb07'); nb08 = load('nb08'); nb09 = load('nb09')
nb10 = load('nb10')
print("✅ Resultados carregados")


## 📊 NB01 — Aprendizado Supervisionado

In [ ]:
data = nb01.get('models', {})
print("NB01 — Aprendizado Supervisionado")
print("-" * 50)
for model, met in data.items():
    if isinstance(met, dict):
        print(f"  {met.get('label','?'):25s} | pAUC@0.1={met.get('pauc01',0):.4f} | "
              f"F1={met.get('f1',0):.4f} | Lat={met.get('latency_ms','-'):.1f}ms | "
              f"Mem={met.get('memory_mb','-'):.2f}MB")


## 📊 NB02 — Não Supervisionado

In [ ]:
data = nb02.get('models', {})
print("NB02 — Não Supervisionado")
print("-" * 50)
for model, met in data.items():
    if isinstance(met, dict):
        print(f"  {met.get('label','?'):25s} | pAUC@0.1={met.get('pauc01',0):.4f} | "
              f"F1={met.get('f1',0):.4f} | Lat={met.get('latency_ms','-')}")


## 📊 NB03 — Baseline Real (4 Modelos)

In [ ]:
data = nb03.get('models', {})
print("NB03 — Benchmark Real")
print("-" * 60)
header = f"{'Modelo':20s} | {'pAUC@0.1':10s} | {'F1':8s} | {'Lat(ms)':10s} | {'Mem(MB)':8s}"
print(header); print("-"*len(header))
for model, met in data.items():
    if isinstance(met, dict):
        lat = met.get('latency_ms')
        mem = met.get('memory_mb')
        print(f"  {met.get('label','?'):20s} | {met.get('pauc01',0):.4f}     | "
              f"{met.get('f1',0):.4f}   | {lat:.2f} ms   | {mem:.3f} MB" if lat and mem
              else f"  {met.get('label','?'):20s} | {met.get('pauc01',0):.4f}")


## 📊 NB04-06 — Representações Espectrais

In [ ]:
print("NB04-06 — Representações Espectrais")
print("-" * 55)
for nb_id, nb_data in [('NB04 Mel', nb04), ('NB05 CWT', nb05), ('NB06 STFT', nb06)]:
    m = nb_data.get('model', {})
    ext = nb_data.get('extraction_ms', '?')
    print(f"  {nb_id:15s} | pAUC={m.get('pauc01',0):.4f} | "
          f"Ext={ext:.1f}ms" if isinstance(ext, float)
          else f"  {nb_id:15s} | pAUC={m.get('pauc01',0):.4f}")


## 📊 NB07-08 — Super-Vetor Híbrido

In [ ]:
print("NB07-08 — Super-Vetor")
for nb_id, nb_data in [('NB07 Mel+MFCC+CWT', nb07), ('NB08 +NMF+Kurt', nb08)]:
    m = nb_data.get('model', {})
    n_feat = m.get('n_features','?')
    print(f"  {nb_id:22s} | pAUC={m.get('pauc01',0):.4f} | Dimensões={n_feat}")


## 📊 NB09-10 — Validação Cross-Domain e Domain Shift

In [ ]:
print("NB09 — Kaggle Bearings (cross-domain)")
m9 = nb09.get('model', {})
print(f"  pAUC@0.1 = {m9.get('pauc01',0):.4f}")

print("\nNB10 — Drive Próprio (domain shift severo)")
m10 = nb10.get('model', {})
drop = nb10.get('domain_shift_drop', 0)
ctrl = m10.get('pauc01_controlled', '?')
print(f"  pAUC@0.1 controlado = {ctrl:.4f}" if isinstance(ctrl, float) else f"  pAUC controlado = {ctrl}")
print(f"  pAUC@0.1 cego (Drive) = {m10.get('pauc01',0):.4f}")
print(f"  Queda = {drop:.4f} pp  ← Motivação das 11 Frentes de Melhoria")


## 🏆 Resumo Fase 1

| Componente | Decisão | Critério |
|-----------|---------|----------|
| XGBoost | ✅ Campeão supervisionado | Melhor pAUC + Edge-ready |
| CNN proxy | 🔴 Descartado como motor | Lat+Mem fora dos limites |
| OC-SVM | 🔴 Descartado | pAUC < 0.80 |
| Autoencoder | 🔴 Conceito mantido via NMF | Lat+Mem fora dos limites |
| Isolation Forest | 🔴 Descartado | pAUC < 0.80 |
| Mel-Spectrogram | ✅ Representação principal | Melhor integração pipeline |
| CWT | 🔴 Descartado como motor | Lat extração esgota orçamento |
| LOSO | ✅ Protocolo adotado | Única medição justa de generalização |
